In [ ]:
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys
import torch

REPO_URL = "https://github.com/ituvtu/reid_investigation.git"
REPO_DIR = Path("/kaggle/working/reid_investigation")


def _run_step(
    step_label: str,
    command: list[str],
    allow_failure: bool = False,
    timeout_seconds: int | None = None,
) -> subprocess.CompletedProcess[str]:
    print(f"[{step_label}] START: {' '.join(command)}")

    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=timeout_seconds)
    except subprocess.TimeoutExpired as error:
        stdout_text = (error.stdout or "").strip()
        stderr_text = (error.stderr or "").strip()

        if stdout_text:
            print(f"[{step_label}] STDOUT (partial):\n{stdout_text}")
        if stderr_text:
            print(f"[{step_label}] STDERR (partial):\n{stderr_text}")

        timeout_message = f"[{step_label}] TIMED OUT after {timeout_seconds} seconds;"
        if allow_failure:
            print(f"{timeout_message} continuing due to allow_failure=True;")
            return subprocess.CompletedProcess(command, 124, stdout_text, stderr_text)
        raise RuntimeError(timeout_message) from error

    if result.stdout.strip():
        print(f"[{step_label}] STDOUT:\n{result.stdout.strip()}")
    if result.stderr.strip():
        print(f"[{step_label}] STDERR:\n{result.stderr.strip()}")

    if result.returncode != 0 and not allow_failure:
        raise RuntimeError(f"[{step_label}] FAILED with code {result.returncode};")

    if result.returncode == 0:
        print(f"[{step_label}] DONE;")
    else:
        print(f"[{step_label}] FAILED but continuing due to allow_failure=True;")

    return result


def _write_requirements_without_dinov2(source_path: Path, output_path: Path) -> None:
    lines = source_path.read_text(encoding="utf-8").splitlines()
    filtered = [line for line in lines if "dinov2" not in line.lower()]
    output_path.write_text("\n".join(filtered) + "\n", encoding="utf-8")


def _is_build_isolation_issue(text: str) -> bool:
    normalized = text.lower()
    markers = (
        "build backend returned an error",
        "no module named 'numpy'",
        "no module named 'wrapt'",
        "no-build-isolation",
    )
    return any(marker in normalized for marker in markers)


def _is_dinov2_solver_conflict(text: str) -> bool:
    normalized = text.lower()
    return "dinov2" in normalized and "unsatisfiable" in normalized


def _install_requirements_resilient(uv_command: list[str], full_requirements: Path) -> None:
    runtime_requirements = Path("/tmp/requirements_stage2_runtime_no_dinov2.txt")
    _write_requirements_without_dinov2(full_requirements, runtime_requirements)

    include_dinov2_optional = os.environ.get("INSTALL_DINOV2_OPTIONAL", "0") == "1"
    requirements_to_install = full_requirements if include_dinov2_optional else runtime_requirements

    if include_dinov2_optional:
        print("[6/9] INSTALL_DINOV2_OPTIONAL=1; using full requirements;")
    else:
        print("[6/9] Using runtime requirements without dinov2 for compatibility;")

    print("[6/9] Dependency installation can take several minutes;")
    install_result = _run_step(
        "6/9",
        uv_command + ["pip", "install", "-r", str(requirements_to_install), "--system"],
        allow_failure=True,
        timeout_seconds=1800,
    )

    if install_result.returncode == 0:
        return

    merged_output = f"{install_result.stdout}\n{install_result.stderr}"

    if _is_dinov2_solver_conflict(merged_output):
        print("[6/9] dinov2 solver conflict detected; retry without dinov2;")
        requirements_to_install = runtime_requirements
        retry_no_dino = _run_step(
            "6/9A",
            uv_command + ["pip", "install", "-r", str(requirements_to_install), "--system", "--no-build-isolation"],
            allow_failure=True,
            timeout_seconds=1800,
        )
        if retry_no_dino.returncode == 0:
            return
        print("[6/9A] uv retry without dinov2 failed; fallback to pip --no-build-isolation;")
        _run_step(
            "6/9B",
            [sys.executable, "-m", "pip", "install", "-r", str(requirements_to_install), "--no-build-isolation"],
            timeout_seconds=1800,
        )
        return

    if _is_build_isolation_issue(merged_output):
        print("[6/9] Build-isolation issue detected; preinstall build deps and retry;")
        _run_step(
            "6/9X",
            [sys.executable, "-m", "pip", "install", "-q", "numpy", "wrapt", "setuptools", "wheel"],
        )

        retry_result = _run_step(
            "6/9Y",
            uv_command + ["pip", "install", "-r", str(requirements_to_install), "--system", "--no-build-isolation"],
            allow_failure=True,
            timeout_seconds=1800,
        )

        if retry_result.returncode == 0:
            return

        retry_merged_output = f"{retry_result.stdout}\n{retry_result.stderr}"
        if _is_dinov2_solver_conflict(retry_merged_output):
            print("[6/9Y] dinov2 solver conflict after retry; switch to requirements without dinov2;")
            requirements_to_install = runtime_requirements

        print("[6/9Y] uv retry failed; fallback to pip --no-build-isolation (can take several minutes);")
        _run_step(
            "6/9Z",
            [sys.executable, "-m", "pip", "install", "-r", str(requirements_to_install), "--no-build-isolation"],
            timeout_seconds=1800,
        )
        return

    raise RuntimeError(f"[6/9] FAILED with code {install_result.returncode};")


def _normalize_soccernet_root(path: Path) -> Path:
    if path.name.lower() in {"train", "valid", "test", "challenge"}:
        return path.parent
    return path


def _collect_soccernet_candidates(input_root: Path) -> list[Path]:
    candidates: list[Path] = []

    known_candidates = [
        Path("/kaggle/input/soccernet-tracking"),
        Path("/kaggle/input/SoccerNet"),
        Path("/kaggle/input/SoccerNet tracking"),
        Path("/kaggle/input/datasets/theoviel/soccernet-tracking"),
        Path("/kaggle/input/datasets/theoviel/soccernet-tracking/train"),
    ]

    for candidate in known_candidates:
        if candidate.exists() and candidate.is_dir():
            candidates.append(candidate)

    if input_root.exists():
        # Limit scan depth to keep startup fast on large Kaggle mounts.
        for child in sorted(input_root.iterdir()):
            if not child.is_dir():
                continue

            if "soccernet" in child.name.lower():
                candidates.append(child)

            for grandchild in sorted(child.iterdir()):
                if grandchild.is_dir() and "soccernet" in grandchild.name.lower():
                    candidates.append(grandchild)

    unique: list[Path] = []
    seen: set[str] = set()
    for path in candidates:
        normalized = _normalize_soccernet_root(path)
        key = str(normalized.resolve())
        if key in seen:
            continue
        unique.append(normalized)
        seen.add(key)

    return unique


def _list_available_splits(root_path: Path) -> list[str]:
    split_names = {"train", "valid", "test", "challenge"}
    if not root_path.exists() or not root_path.is_dir():
        return []
    return sorted(
        child.name
        for child in root_path.iterdir()
        if child.is_dir() and child.name.lower() in split_names
    )


def _check_torchreid_runtime() -> tuple[bool, str]:
    try:
        importlib.invalidate_caches()
        if "torchreid" in sys.modules:
            del sys.modules["torchreid"]
        torchreid_module = importlib.import_module("torchreid")
    except Exception as error:
        return False, f"import torchreid failed: {error}"

    models_api = getattr(torchreid_module, "models", None)
    if models_api is None or not hasattr(models_api, "build_model"):
        return False, "torchreid.models.build_model is unavailable (incompatible package variant)"

    try:
        from torchreid.utils import load_pretrained_weights as _  # noqa: F401
    except Exception as error:
        return False, f"import torchreid.utils.load_pretrained_weights failed: {error}"

    return True, ""


def _ensure_torchreid_runtime() -> None:
    ok, error_text = _check_torchreid_runtime()
    if ok:
        print("[6/9C] torchreid runtime check: OK;")
        return

    print(f"[6/9C] torchreid runtime check failed: {error_text};")
    print("[6/9C] Applying torchreid fallback installation;")

    _run_step(
        "6/9C",
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "yacs",
            "tensorboard",
            "future",
            "gdown",
        ],
        timeout_seconds=900,
    )
    _run_step(
        "6/9D",
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchreid"],
        allow_failure=True,
        timeout_seconds=600,
    )
    _run_step(
        "6/9E",
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--force-reinstall",
            "--no-deps",
            "git+https://github.com/KaiyangZhou/deep-person-reid.git",
        ],
        timeout_seconds=1200,
    )

    ok_after, error_text_after = _check_torchreid_runtime()
    if not ok_after:
        raise RuntimeError(
            "[6/9E] torchreid is still unavailable after fallback installation; "
            f"details: {error_text_after};"
        )

    print("[6/9E] torchreid runtime check: OK after fallback;")


print("[1/9] Bootstrap started;")
print(f"Current working directory: {Path.cwd()};")
print(f"Repository target path: {REPO_DIR};")

print("[2/9] Repository clone check;")
if not REPO_DIR.exists():
    _run_step("2/9", ["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    print("[2/9] Repository already exists - clone skipped;")

skip_git_pull = os.environ.get("REID_SKIP_GIT_PULL", "0") == "1"
if skip_git_pull:
    print("[2/9A] Git pull skipped because REID_SKIP_GIT_PULL=1;")
else:
    print("[2/9A] Sync repository with remote (git pull --ff-only);")
    pull_result = _run_step(
        "2/9A",
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        allow_failure=True,
        timeout_seconds=300,
    )
    if pull_result.returncode != 0:
        print("[2/9A] Pull failed, continuing with local repository state;")

print("[3/9] Change directory;")
os.chdir(REPO_DIR)
print(f"[3/9] Current working directory: {Path.cwd()};")

print("[4/9] Install uv package manager;")
_run_step("4/9", [sys.executable, "-m", "pip", "install", "-q", "uv"], timeout_seconds=300)

print("[5/9] Resolve uv command;")
uv_cli = shutil.which("uv")
if uv_cli is None:
    uv_command = [sys.executable, "-m", "uv"]
    print("[5/9] Using fallback: python -m uv;")
else:
    uv_command = [uv_cli]
    print(f"[5/9] Using uv executable: {uv_cli};")

print("[6/9] Install Python dependencies from requirements.txt;")
requirements_path = Path("requirements.txt")
_install_requirements_resilient(uv_command, requirements_path)

_ensure_torchreid_runtime()

print("[7/9] Inject repository path into sys.path;")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"[7/9] Path injected: {str(REPO_DIR) in sys.path};")

print("[8/9] Detect SoccerNet dataset paths (non-recursive scan);")
input_root = Path("/kaggle/input")
soccernet_roots = _collect_soccernet_candidates(input_root)

print(f"[8/9] SoccerNet candidates found: {len(soccernet_roots)};")
if soccernet_roots:
    for path in soccernet_roots[:10]:
        print(f"- {path};")

    selected_root = soccernet_roots[0]
    os.environ.setdefault("SOCCERNET_ROOT_DIR", str(selected_root))
    print(f"[8/9] Selected SOCCERNET_ROOT_DIR: {os.environ['SOCCERNET_ROOT_DIR']};")

    available_splits = _list_available_splits(Path(os.environ["SOCCERNET_ROOT_DIR"]))
    print(f"[8/9] Available splits at selected root: {available_splits};")
else:
    print(f"[8/9] No SoccerNet candidate found under: {input_root};")

os.environ.setdefault("STAGE2_ALLOWED_SPLITS", "train,test")
print(f"[8/9] STAGE2_ALLOWED_SPLITS: {os.environ['STAGE2_ALLOWED_SPLITS']};")

print("[9/9] GPU and environment summary;")
gpu_detected = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_detected else "No GPU detected"

print(f"GPU detected: {gpu_detected};")
print(f"GPU name: {gpu_name};")
print(f"SOCCERNET_ROOT_DIR: {os.environ.get('SOCCERNET_ROOT_DIR', '<not-set>')};")
print("[9/9] Bootstrap completed;")

In [ ]:
from pathlib import Path
import os

DEFAULT_REPO_URL = "https://github.com/ituvtu/reid_investigation.git"
os.environ.setdefault("REID_REPO_URL", DEFAULT_REPO_URL)

# Optional overrides for Kaggle runtime:
# os.environ["SOCCERNET_ROOT_DIR"] = "/kaggle/input/datasets/theoviel/soccernet-tracking"
# os.environ["SOCCERNET_PASSWORD"] = "<password_if_required>"

print("Working directory:", Path.cwd())
print("models exists:", os.path.exists("models"))
print("configs exists:", os.path.exists("configs"))
print("REID_REPO_URL:", os.environ.get("REID_REPO_URL", ""))

In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys
import warnings


def _run_or_fail(command: list[str]) -> None:
    print("$", " ".join(command))
    result = subprocess.run(command, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(command)}")


def _check_torchreid_runtime() -> tuple[bool, str]:
    try:
        importlib.invalidate_caches()
        if "torchreid" in sys.modules:
            del sys.modules["torchreid"]
        torchreid_module = importlib.import_module("torchreid")
    except Exception as error:
        return False, f"import torchreid failed: {error}"

    models_api = getattr(torchreid_module, "models", None)
    if models_api is None or not hasattr(models_api, "build_model"):
        return False, "torchreid.models.build_model is unavailable"

    try:
        from torchreid.utils import load_pretrained_weights as _  # noqa: F401
    except Exception as error:
        return False, f"import torchreid.utils.load_pretrained_weights failed: {error}"

    return True, ""


runtime_ok, runtime_error = _check_torchreid_runtime()
if not runtime_ok:
    print(f"torchreid runtime mismatch detected: {runtime_error};")
    print("Applying fallback repair for deep-person-reid;")
    _run_or_fail([sys.executable, "-m", "pip", "install", "yacs", "tensorboard", "future", "gdown"])
    _run_or_fail([sys.executable, "-m", "pip", "uninstall", "-y", "torchreid"])
    _run_or_fail(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--force-reinstall",
            "--no-deps",
            "git+https://github.com/KaiyangZhou/deep-person-reid.git",
        ]
    )
    runtime_ok, runtime_error = _check_torchreid_runtime()
    if not runtime_ok:
        raise RuntimeError(f"torchreid runtime is still invalid after repair: {runtime_error};")

print("torchreid runtime check: OK;")

from models.detectors import YOLODetector
from models.reid import OSNetReID
from models.trackers import ByteTrackTracker
from utils import load_stage2_reid_config, stage2_component_mappings

config = load_stage2_reid_config("configs/stage2_reid.yaml")
detector_config, tracker_config, reid_config = stage2_component_mappings(config)

requested_weights = str(detector_config.get("weights_path", "")).strip()
requested_model = str(detector_config.get("model_name", "")).strip()
weights_path = Path(requested_weights) if requested_weights else None

if requested_model.lower().startswith("yolo26"):
    detector_config["weights_path"] = requested_model
    print(f"YOLOv26 library mode enabled: source={requested_model};")
elif requested_weights and weights_path is not None and not weights_path.exists():
    raise FileNotFoundError(
        f"Weights file not found: {requested_weights}; provide a valid path or use a model name source."
    )

detector = YOLODetector.from_config(detector_config)
detector.load()
detector.warmup(image_size=(640, 640), runs=1)

shared_device = detector.device
tracker_config["device"] = shared_device
reid_config["device"] = shared_device

tracker = ByteTrackTracker.from_config(tracker_config)
tracker.initialize()

reid_model = OSNetReID.from_config(reid_config)
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=FutureWarning, message='.*weights_only=False.*')
    reid_model.load()

print(f"Detector device: {detector.device}")
print(f"Tracker device: {tracker.device}")
print(f"ReID device: {reid_model.device}")
print(f"Experiment: {config.experiment_name}")

if not str(detector.device).startswith("cuda"):
    print("Warning: CUDA was not found - running on CPU;")

In [ ]:
import csv
import os
from pathlib import Path

import cv2
from core.base_tracker import Track
from utils import SoccerNetLoader

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}
KNOWN_SPLITS = ('train', 'test', 'challenge', 'valid')

if config.dataset is None or config.dataset.soccernet is None:
    raise ValueError("Missing dataset.soccernet configuration in configs/stage2_reid.yaml")

soccernet_cfg = config.dataset.soccernet
override_root = os.environ.get("SOCCERNET_ROOT_DIR", "").strip()
override_password = os.environ.get("SOCCERNET_PASSWORD", "").strip() or soccernet_cfg.password

requested_source_count = int(os.environ.get('STAGE2_SOURCE_COUNT', '10'))
if requested_source_count < 1:
    raise ValueError('STAGE2_SOURCE_COUNT must be >= 1;')

allowed_splits_env = os.environ.get('STAGE2_ALLOWED_SPLITS', 'train,test')
allowed_splits = {item.strip().lower() for item in allowed_splits_env.split(',') if item.strip()}
if not allowed_splits:
    raise ValueError('STAGE2_ALLOWED_SPLITS must define at least one split;')

use_stage1_source_list = os.environ.get('STAGE2_USE_STAGE1_SOURCE_LIST', '1') == '1'
stage1_source_list_path = Path(os.environ.get('STAGE1_SOURCE_LIST_PATH', 'reports/stage1/stage1_metrics_by_source.csv'))

candidate_roots_raw = [
    Path(override_root).expanduser() if override_root else None,
    Path(soccernet_cfg.root_dir).expanduser(),
    Path('/kaggle/input/SoccerNet tracking'),
    Path('/kaggle/input/soccernet-tracking'),
    Path('/kaggle/input/datasets/theoviel/soccernet-tracking'),
]

candidate_roots: list[Path] = []
seen_roots: set[str] = set()
for candidate in candidate_roots_raw:
    if candidate is None:
        continue
    key = str(candidate)
    if key in seen_roots:
        continue
    candidate_roots.append(candidate)
    seen_roots.add(key)

print('Dataset root candidates (in priority order):')
for index, candidate in enumerate(candidate_roots, start=1):
    exists_text = 'exists' if candidate.exists() else 'missing'
    print(f"{index}. {candidate} [{exists_text}];")

resolved_root: Path | None = None
for candidate in candidate_roots:
    if candidate.exists():
        resolved_root = candidate
        break

if resolved_root is None:
    rendered_candidates = [str(path) for path in candidate_roots]
    raise RuntimeError(
        'SoccerNet root directory was not found. Checked candidates: ' + ', '.join(rendered_candidates) + ';'
    )

target_root = resolved_root
dataset_mapping = {
    'root_dir': str(target_root),
    'subset': soccernet_cfg.subset,
    'split': list(soccernet_cfg.split),
    'password': override_password,
    'auto_download': False,
}

print('Dataset configuration:')
print(f"root_dir: {dataset_mapping['root_dir']};")
print(f"subset: {dataset_mapping['subset']};")
print(f"split from config: {dataset_mapping['split']};")
print('auto_download: False (Kaggle dataset already extracted);')
print(f"Requested source count: {requested_source_count};")
print(f"Allowed splits: {sorted(allowed_splits)};")
print(f"Use stage1 source list: {use_stage1_source_list};")
print(f"Stage1 source list path: {stage1_source_list_path};")

soccernet_loader = SoccerNetLoader.from_config(dataset_mapping)
dataset_root = target_root


def _list_split_directories(root: Path) -> dict[str, Path]:
    split_dirs: dict[str, Path] = {}
    for split_name in KNOWN_SPLITS:
        split_path = root / split_name
        if split_path.exists() and split_path.is_dir():
            split_dirs[split_name] = split_path
    return split_dirs


def _iter_split_sequence_dirs(split_path: Path) -> list[Path]:
    sequence_dirs: list[Path] = []
    for sequence_dir in sorted(split_path.iterdir()):
        if sequence_dir.is_dir() and (sequence_dir / 'img1').is_dir():
            sequence_dirs.append(sequence_dir)
    return sequence_dirs


def _count_sequences_in_split(split_path: Path) -> int:
    return len(_iter_split_sequence_dirs(split_path))


def _load_mot_table_tracks(path: Path) -> dict[int, list[Track]]:
    tracks_by_frame: dict[int, list[Track]] = {}
    with path.open('r', encoding='utf-8', newline='') as file:
        reader = csv.reader(file)
        for row in reader:
            if len(row) < 6:
                continue
            try:
                frame_id = int(float(row[0]))
                track_id = int(float(row[1]))
                x = float(row[2])
                y = float(row[3])
                w = float(row[4])
                h = float(row[5])
                confidence = float(row[6]) if len(row) > 6 and row[6] != '' else 1.0
            except Exception:
                continue

            tracks_by_frame.setdefault(frame_id, []).append(
                Track(
                    track_id=track_id,
                    bbox_xyxy=(x, y, x + w, y + h),
                    confidence=confidence,
                    class_id=0,
                    embedding=None,
                )
            )
    return tracks_by_frame


def iter_image_sequence_frames(image_files: list[Path], max_frames: int | None = None, stride: int = 1):
    produced = 0
    for image_path in image_files:
        frame_id = None
        try:
            frame_id = int(image_path.stem)
        except Exception:
            continue

        if frame_id % stride != 0:
            continue

        frame = cv2.imread(str(image_path))
        if frame is None:
            continue

        yield frame_id, frame
        produced += 1

        if max_frames is not None and produced >= max_frames:
            break


def _choose_sequence_annotation(sequence_root: Path) -> Path | None:
    gt_dir = sequence_root / 'gt'
    candidates: list[Path] = []

    if gt_dir.exists():
        candidates.extend(sorted(gt_dir.glob('*.json')))
        candidates.extend(sorted(gt_dir.glob('*.csv')))
        candidates.extend(sorted(gt_dir.glob('*.txt')))

    candidates.extend(sorted(sequence_root.glob('*.json')))
    candidates.extend(sorted(sequence_root.glob('*.csv')))
    candidates.extend(sorted(sequence_root.glob('*.txt')))

    filtered_candidates = [
        path for path in candidates
        if path.is_file() and ('gt' in path.name.lower() or 'tracking' in path.name.lower() or path.suffix.lower() == '.json')
    ]

    if filtered_candidates:
        return filtered_candidates[0]
    return candidates[0] if candidates else None


def _load_annotation_tracks(annotation_path: Path) -> dict[int, list[Track]]:
    if annotation_path.suffix.lower() == '.json':
        annotation_payload = soccernet_loader.load_tracking_annotations(annotation_path=annotation_path)
        return soccernet_loader.map_tracking_annotations_to_tracks(annotation_payload)

    if annotation_path.suffix.lower() in {'.csv', '.txt'}:
        return _load_mot_table_tracks(annotation_path)

    raise RuntimeError(f'Unsupported annotation format: {annotation_path.suffix}')


def _split_name_from_source(source_name: str) -> str:
    source_parts = [part.lower() for part in Path(source_name).parts]
    for split_name in KNOWN_SPLITS:
        if split_name in source_parts:
            return split_name
    return 'unknown'


def _normalize_source_name(source_name: str) -> str:
    normalized = source_name.replace('\\', '/').strip().lstrip('./')
    if normalized.startswith('tracking/'):
        normalized = normalized[len('tracking/'): ]
    return normalized


def _parse_stage1_source_list(csv_path: Path, limit: int) -> list[str]:
    ordered_sources: list[str] = []
    seen: set[str] = set()

    with csv_path.open('r', encoding='utf-8', newline='') as file:
        reader = csv.DictReader(file)
        for row in reader:
            source_value = str(row.get('Source', '')).strip()
            if not source_value:
                continue
            normalized = _normalize_source_name(source_value)
            if normalized in seen:
                continue
            ordered_sources.append(normalized)
            seen.add(normalized)
            if len(ordered_sources) >= limit:
                break

    return ordered_sources


def _resolve_sequence_dir_from_name(root: Path, split_directories: dict[str, Path], normalized_source: str) -> Path | None:
    parts = [part for part in normalized_source.split('/') if part]
    if len(parts) >= 2 and parts[0].lower() in split_directories:
        split_name = parts[0].lower()
        sequence_name = parts[1]
        candidate = split_directories[split_name] / sequence_name
        if candidate.exists() and candidate.is_dir():
            return candidate

    if parts:
        sequence_name = parts[-1]
        for split_name, split_path in split_directories.items():
            candidate = split_path / sequence_name
            if candidate.exists() and candidate.is_dir():
                return candidate

    return None


def _build_source_record(root: Path, sequence_root: Path) -> dict | None:
    img_dir = sequence_root / 'img1'
    if not img_dir.exists() or not img_dir.is_dir():
        return None

    image_files = sorted(
        [path for path in img_dir.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS],
        key=lambda path: path.name,
    )
    if not image_files:
        return None

    annotation_path = _choose_sequence_annotation(sequence_root)
    if annotation_path is None:
        return None

    gt_tracks = _load_annotation_tracks(annotation_path)
    if not gt_tracks:
        return None

    try:
        source_name = sequence_root.relative_to(root).as_posix()
    except Exception:
        source_name = sequence_root.name

    return {
        'source_name': source_name,
        'frame_source_kind': 'image_sequence',
        'source_path': img_dir,
        'image_files': image_files,
        'annotation_path': annotation_path,
        'gt_tracks_by_frame': gt_tracks,
    }


split_directories = _list_split_directories(dataset_root)
if not split_directories:
    raise RuntimeError(f'No split directories found under dataset root: {dataset_root};')

print('Detected split directories and sequence counts:')
for split_name, split_path in split_directories.items():
    sequence_count = _count_sequences_in_split(split_path)
    split_tag = 'allowed' if split_name in allowed_splits else 'skipped'
    print(f"- {split_name}: {split_path} | sequences with img1={sequence_count} | {split_tag};")

if not any(split in split_directories for split in allowed_splits):
    raise RuntimeError(
        f"None of allowed splits {sorted(allowed_splits)} exist under dataset root: {dataset_root};"
    )

print('Dataset format summary: image sequences in img1/ with per-sequence annotations; video files are not required;')

preferred_source_names: list[str] = []
if use_stage1_source_list and stage1_source_list_path.exists():
    preferred_source_names = _parse_stage1_source_list(stage1_source_list_path, requested_source_count)
    print(f"Loaded preferred source names from stage1 report: {len(preferred_source_names)};")
    for index, name in enumerate(preferred_source_names, start=1):
        print(f"- stage1 order {index}: {name};")
elif use_stage1_source_list:
    print('Stage1 source list file is missing; fallback to deterministic discovery by sorted source_name;')
else:
    print('Stage1 source list usage disabled; fallback to deterministic discovery by sorted source_name;')

available_sequence_sources: list[dict] = []
selected_source_names: set[str] = set()

if preferred_source_names:
    print('Selecting sources using stage1 order...')
    for preferred_name in preferred_source_names:
        sequence_dir = _resolve_sequence_dir_from_name(dataset_root, split_directories, preferred_name)
        if sequence_dir is None:
            print(f"- skip (not found): {preferred_name};")
            continue

        try:
            source_record = _build_source_record(dataset_root, sequence_dir)
        except Exception as error:
            print(f"- skip (load failed): {preferred_name} | {error};")
            continue

        if source_record is None:
            print(f"- skip (invalid sequence): {preferred_name};")
            continue

        split_name = _split_name_from_source(str(source_record['source_name']))
        if split_name not in allowed_splits:
            print(f"- skip (split filtered): {preferred_name};")
            continue

        source_key = str(source_record['source_name'])
        if source_key in selected_source_names:
            continue

        available_sequence_sources.append(source_record)
        selected_source_names.add(source_key)
        print(
            f"- accepted {len(available_sequence_sources)}/{requested_source_count}: {source_key} | "
            f"images={len(source_record['image_files'])} | gt_frames={len(source_record['gt_tracks_by_frame'])};"
        )

        if len(available_sequence_sources) >= requested_source_count:
            break

if len(available_sequence_sources) < requested_source_count:
    print(
        f"Falling back to deterministic discovery for remaining sources: "
        f"need {requested_source_count - len(available_sequence_sources)} more;"
    )

    candidate_sequence_dirs: list[Path] = []
    for split_name in sorted(allowed_splits):
        split_path = split_directories.get(split_name)
        if split_path is None:
            continue
        candidate_sequence_dirs.extend(_iter_split_sequence_dirs(split_path))

    candidate_sequence_dirs = sorted(
        candidate_sequence_dirs,
        key=lambda path: path.relative_to(dataset_root).as_posix(),
    )

    for sequence_dir in candidate_sequence_dirs:
        if len(available_sequence_sources) >= requested_source_count:
            break

        try:
            source_record = _build_source_record(dataset_root, sequence_dir)
        except Exception as error:
            print(f"- fallback skip (load failed): {sequence_dir.name} | {error};")
            continue

        if source_record is None:
            continue

        source_key = str(source_record['source_name'])
        if source_key in selected_source_names:
            continue

        available_sequence_sources.append(source_record)
        selected_source_names.add(source_key)
        print(
            f"- fallback accepted {len(available_sequence_sources)}/{requested_source_count}: {source_key} | "
            f"images={len(source_record['image_files'])} | gt_frames={len(source_record['gt_tracks_by_frame'])};"
        )

if len(available_sequence_sources) < requested_source_count:
    raise RuntimeError(
        f"Requested {requested_source_count} sources but only found {len(available_sequence_sources)} valid sequences;"
    )

evaluation_sources = available_sequence_sources[:requested_source_count]

# Compatibility variables for downstream cells and ad-hoc checks.
primary_source = evaluation_sources[0]
frame_source_kind = primary_source['frame_source_kind']
sequence_image_files = primary_source['image_files']
video_path = primary_source['source_path']
annotation_path = primary_source['annotation_path']
gt_tracks_by_frame = primary_source['gt_tracks_by_frame']

print(f'Selected sources for evaluation: {len(evaluation_sources)};')
for index, source in enumerate(evaluation_sources, start=1):
    gt_frames = len(source['gt_tracks_by_frame'])
    image_count = len(source['image_files'])
    split_name = _split_name_from_source(str(source['source_name']))
    print(
        f"{index}. split={split_name} | {source['source_name']} | images={image_count} | "
        f"gt_frames={gt_frames} | ann={source['annotation_path']};"
    )

In [ ]:
import importlib
import utils.metrics as metrics_module

importlib.reload(metrics_module)
from utils import compute_mot_id_metrics

print('Reloaded utils.metrics with NumPy 2 compatibility shim;')

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

from core.base_tracker import Track
from utils import compute_mot_id_metrics

# Add compatibility for NumPy 2.x;
if not hasattr(np, 'asfarray'):
    def _np_asfarray_compat(values, dtype=float):
        return np.asarray(values, dtype=dtype)

    np.asfarray = _np_asfarray_compat

try:
    import cv2
except Exception:
    cv2 = None
    print('Warning: OpenCV is unavailable - visualization will be limited;')

if 'evaluation_sources' not in globals() or not evaluation_sources:
    raise RuntimeError('evaluation_sources is missing - run Cell 4 first;')

if 'reid_model' not in globals():
    raise RuntimeError('reid_model is missing - run Cell 3 first;')

if 'tracker' not in globals():
    raise RuntimeError('tracker is missing - run Cell 3 first;')

max_frames_per_source = int(os.environ.get('STAGE2_MAX_FRAMES', '120'))
frame_stride = int(os.environ.get('STAGE2_FRAME_STRIDE', '2'))
iou_threshold = float(os.environ.get('STAGE2_IOU_THRESHOLD', '0.5'))
preview_enabled = os.environ.get('STAGE2_ENABLE_PREVIEW', '1') == '1'

if max_frames_per_source < 1:
    raise ValueError('STAGE2_MAX_FRAMES must be >= 1;')
if frame_stride < 1:
    raise ValueError('STAGE2_FRAME_STRIDE must be >= 1;')

association_alpha_value = float(getattr(tracker, 'association_alpha', tracker_config.get('association_alpha', 0.5)))
default_input_size = reid_config.get('input_size', [256, 128])
reid_input_height = int(getattr(reid_model, 'input_height', int(default_input_size[0])))
reid_input_width = int(getattr(reid_model, 'input_width', int(default_input_size[1])))

print(f'Sources selected: {len(evaluation_sources)};')
print(f'Max frames per source: {max_frames_per_source};')
print(f'Frame stride: {frame_stride};')
print(f'Estimated max processed frames: {len(evaluation_sources) * max_frames_per_source};')
print(f'Current association alpha parameter: {association_alpha_value:.3f};')
print(f'ReID crop size: {reid_input_height}x{reid_input_width};')
print(f'Preview enabled: {preview_enabled};')


def _offset_tracks_by_frame_and_id(
    tracks_by_frame: dict[int, list[Track]],
    frame_offset: int,
    track_offset: int,
) -> dict[int, list[Track]]:
    remapped: dict[int, list[Track]] = {}
    for frame_id, tracks in tracks_by_frame.items():
        remapped_frame = int(frame_id) + frame_offset
        remapped[remapped_frame] = [
            Track(
                track_id=int(track.track_id) + track_offset,
                bbox_xyxy=track.bbox_xyxy,
                confidence=float(track.confidence),
                class_id=int(track.class_id),
                embedding=track.embedding,
            )
            for track in tracks
        ]
    return remapped


def _extract_crops_for_reid(frame: np.ndarray, detections) -> list[np.ndarray]:
    # Clip bounding boxes to frame boundaries and resize crops to the OSNet input size;
    if frame is None or frame.size == 0:
        return [np.zeros((reid_input_height, reid_input_width, 3), dtype=np.uint8) for _ in detections]

    frame_height, frame_width = frame.shape[:2]
    if frame_height < 1 or frame_width < 1:
        return [np.zeros((reid_input_height, reid_input_width, 3), dtype=np.uint8) for _ in detections]

    crops: list[np.ndarray] = []
    for detection in detections:
        x1, y1, x2, y2 = detection.bbox_xyxy

        left = int(np.clip(np.floor(x1), 0, frame_width - 1))
        top = int(np.clip(np.floor(y1), 0, frame_height - 1))
        right = int(np.clip(np.ceil(x2), left + 1, frame_width))
        bottom = int(np.clip(np.ceil(y2), top + 1, frame_height))

        crop = frame[top:bottom, left:right]
        if cv2 is None or crop.size == 0:
            resized_crop = np.zeros((reid_input_height, reid_input_width, 3), dtype=np.uint8)
        else:
            resized_crop = cv2.resize(
                crop,
                (reid_input_width, reid_input_height),
                interpolation=cv2.INTER_LINEAR,
            )

        crops.append(resized_crop)

    return crops


preview_frames: list[tuple[str, int, np.ndarray]] = []
per_source_metrics_state: list[dict] = []
combined_ground_truth: dict[int, list[Track]] = {}
combined_predictions: dict[int, list[Track]] = {}
evaluation_started_at = time.perf_counter()

for source_index, source in enumerate(evaluation_sources):
    source_name = str(source['source_name'])
    source_kind = str(source['frame_source_kind'])
    source_gt_tracks_by_frame = source['gt_tracks_by_frame']
    source_label = f"{source_index + 1}/{len(evaluation_sources)}"

    print(f"[Source {source_label}] start: {source_name} ({source_kind});")

    if source_kind == 'video':
        frame_iterator = soccernet_loader.iter_video_frames(
            video_path=source['source_path'],
            max_frames=max_frames_per_source,
            stride=frame_stride,
        )
    else:
        frame_iterator = iter_image_sequence_frames(
            image_files=source['image_files'],
            max_frames=max_frames_per_source,
            stride=frame_stride,
        )

    tracker.initialize()

    source_latencies: list[float] = []
    source_predictions_by_frame: dict[int, list[Track]] = {}

    for frame_index, frame in frame_iterator:
        frame_start = time.perf_counter()
        detections = detector.predict(frame)

        if detections:
            crops = _extract_crops_for_reid(frame, detections)
            embeddings = reid_model.extract(crops)
        else:
            embeddings = None

        tracks = tracker.update(detections=detections, embeddings=embeddings, frame_index=frame_index)
        source_latencies.append(time.perf_counter() - frame_start)
        source_predictions_by_frame[frame_index] = tracks

        if preview_enabled and len(preview_frames) < 3:
            vis_frame = frame.copy()
            if cv2 is not None:
                cv2.putText(
                    vis_frame,
                    f'alpha={association_alpha_value:.2f}',
                    (12, 24),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 255, 255),
                    2,
                    cv2.LINE_AA,
                )
                for track in tracks:
                    x1, y1, x2, y2 = [int(value) for value in track.bbox_xyxy]
                    cv2.rectangle(vis_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(
                        vis_frame,
                        f'ID {track.track_id}',
                        (x1, max(44, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (255, 255, 255),
                        2,
                        cv2.LINE_AA,
                    )
            preview_frames.append((source_name, int(frame_index), vis_frame))

    if not source_predictions_by_frame:
        raise RuntimeError(f'No frames were processed for source: {source_name};')

    source_metric_values = compute_mot_id_metrics(
        ground_truth_by_frame=source_gt_tracks_by_frame,
        predictions_by_frame=source_predictions_by_frame,
        iou_threshold=iou_threshold,
    )

    source_fps = (1.0 / float(np.mean(source_latencies))) if source_latencies else 0.0
    processed_frames = len(source_predictions_by_frame)

    per_source_metrics_state.append(
        {
            'Source': source_name,
            'MOTA': float(source_metric_values['MOTA']),
            'IDF1': float(source_metric_values['IDF1']),
            'ID Swaps': int(source_metric_values['ID Swaps']),
            'FPS': float(source_fps),
            'Processed Frames': int(processed_frames),
            'Association Alpha': float(association_alpha_value),
        }
    )

    # Isolate ID spaces across sources;
    frame_offset = (source_index + 1) * 1_000_000
    track_offset = (source_index + 1) * 100_000

    combined_ground_truth.update(
        _offset_tracks_by_frame_and_id(source_gt_tracks_by_frame, frame_offset=frame_offset, track_offset=track_offset)
    )
    combined_predictions.update(
        _offset_tracks_by_frame_and_id(source_predictions_by_frame, frame_offset=frame_offset, track_offset=track_offset)
    )

    print(
        f"[Source {source_label}] done: processed_frames={processed_frames} | "
        f"fps={source_fps:.2f} | mota={float(source_metric_values['MOTA']):.4f} | "
        f"idf1={float(source_metric_values['IDF1']):.4f};"
    )

if len(per_source_metrics_state) != len(evaluation_sources):
    raise RuntimeError('Not all selected sources produced valid metrics;')

aggregate_metric_values = compute_mot_id_metrics(
    ground_truth_by_frame=combined_ground_truth,
    predictions_by_frame=combined_predictions,
    iou_threshold=iou_threshold,
)

average_fps = float(np.mean([item['FPS'] for item in per_source_metrics_state])) if per_source_metrics_state else 0.0

metrics_state = {
    'MOTA': float(aggregate_metric_values['MOTA']),
    'IDF1': float(aggregate_metric_values['IDF1']),
    'ID Swaps': int(aggregate_metric_values['ID Swaps']),
    'FPS': float(average_fps),
    'Sources Evaluated': int(len(per_source_metrics_state)),
    'Association Alpha': float(association_alpha_value),
}

preview_count = len(preview_frames)
if preview_enabled and preview_count > 0:
    fig, axes = plt.subplots(1, preview_count, figsize=(5 * preview_count, 4))
    if preview_count == 1:
        axes = [axes]

    for index in range(preview_count):
        source_name, frame_index, frame_to_show = preview_frames[index]
        if cv2 is not None:
            frame_to_show = cv2.cvtColor(frame_to_show, cv2.COLOR_BGR2RGB)
        axes[index].imshow(frame_to_show)
        axes[index].set_title(f'{source_name} - frame {frame_index} - alpha {association_alpha_value:.2f}')
        axes[index].axis('off')

    plt.tight_layout()
elif not preview_enabled:
    print('Preview plotting skipped because STAGE2_ENABLE_PREVIEW=0;')

evaluation_seconds = time.perf_counter() - evaluation_started_at
print('Status: per-source and aggregate metric computation completed;')
print(f"Sources evaluated: {metrics_state['Sources Evaluated']};")
print(f"Evaluation runtime: {evaluation_seconds:.2f}s;")

In [ ]:
from pathlib import Path
import shutil

import pandas as pd

if 'metrics_state' not in globals():
    raise RuntimeError('metrics_state is missing - run Cell 6 first;')
if 'per_source_metrics_state' not in globals():
    raise RuntimeError('per_source_metrics_state is missing - run Cell 6 first;')

aggregate_rows = [
    {'Metric': 'MOTA', 'Value': float(metrics_state['MOTA'])},
    {'Metric': 'IDF1', 'Value': float(metrics_state['IDF1'])},
    {'Metric': 'ID Swaps', 'Value': int(metrics_state['ID Swaps'])},
    {'Metric': 'FPS (avg across sources)', 'Value': float(metrics_state['FPS'])},
    {'Metric': 'Sources Evaluated', 'Value': int(metrics_state['Sources Evaluated'])},
    {'Metric': 'Association Alpha', 'Value': float(metrics_state['Association Alpha'])},
]

aggregate_metrics_table = pd.DataFrame(aggregate_rows)
per_source_metrics_table = pd.DataFrame(per_source_metrics_state)

aggregate_display = aggregate_metrics_table.copy()
aggregate_display['Value'] = aggregate_display['Value'].apply(
    lambda value: f'{value:.3f}' if isinstance(value, float) else value
)

per_source_display = per_source_metrics_table.copy()
for column in ['MOTA', 'IDF1', 'FPS', 'Association Alpha']:
    if column in per_source_display.columns:
        per_source_display[column] = per_source_display[column].map(lambda value: f'{float(value):.3f}')

print('Metrics: aggregate summary across selected sources;')
display(aggregate_display)

print('Metrics: per-source breakdown;')
display(per_source_display)

# Save to multiple Kaggle-compatible output locations so downstream download cells can always resolve files.
candidate_output_dirs = [
    Path.cwd() / 'outputs',
    Path('/kaggle/working/reid_investigation/outputs'),
    Path('/kaggle/working/outputs'),
]

output_dirs: list[Path] = []
seen_output_dirs: set[str] = set()
for candidate_dir in candidate_output_dirs:
    resolved_dir = candidate_dir.resolve()
    key = str(resolved_dir)
    if key in seen_output_dirs:
        continue
    resolved_dir.mkdir(parents=True, exist_ok=True)
    output_dirs.append(resolved_dir)
    seen_output_dirs.add(key)

tables_by_filename: dict[str, pd.DataFrame] = {
    'stage2_metrics.csv': aggregate_metrics_table,
    'stage2_metrics_by_source.csv': per_source_metrics_table,
}

saved_files: dict[str, list[Path]] = {name: [] for name in tables_by_filename}
for out_dir in output_dirs:
    for file_name, metrics_table in tables_by_filename.items():
        out_path = out_dir / file_name
        metrics_table.to_csv(out_path, index=False)

        if not out_path.exists():
            raise RuntimeError(f'Failed to save file: {out_path};')
        if out_path.stat().st_size <= 0:
            raise RuntimeError(f'Saved file is empty: {out_path};')

        saved_files[file_name].append(out_path)

# Mirror files to a dedicated directory used by Kaggle UI and /files links.
export_dir = Path('/kaggle/working/stage2_exports')
export_dir.mkdir(parents=True, exist_ok=True)

preferred_primary_dir = (Path.cwd() / 'outputs').resolve()
for file_name in tables_by_filename:
    preferred_source = preferred_primary_dir / file_name
    source_path = preferred_source if preferred_source.exists() else saved_files[file_name][0]
    target_path = export_dir / file_name
    shutil.copy2(source_path, target_path)

    if not target_path.exists() or target_path.stat().st_size <= 0:
        raise RuntimeError(f'Failed to mirror file into export directory: {target_path};')

print('Saved aggregate metrics CSV and per-source metrics CSV to:')
for file_name, paths in saved_files.items():
    for path in paths:
        print(f'- {file_name}: {path} | bytes={path.stat().st_size};')

print(f'Export mirror directory: {export_dir};')
for file_name in tables_by_filename:
    mirrored_path = export_dir / file_name
    print(f'- mirrored: {mirrored_path} | bytes={mirrored_path.stat().st_size};')

In [ ]:
from pathlib import Path
import shutil
from IPython.display import HTML, display

print('Runtime mode: Kaggle-only file export;')

required_names = [
    'stage2_metrics.csv',
    'stage2_metrics_by_source.csv',
]

search_roots = [
    Path.cwd(),
    Path('/kaggle/working/reid_investigation'),
    Path('/kaggle/working'),
]


def _resolve_metrics_file(file_name: str) -> Path | None:
    direct_candidates = [
        Path('outputs') / file_name,
        Path.cwd() / 'outputs' / file_name,
        Path('/kaggle/working/reid_investigation/outputs') / file_name,
        Path('/kaggle/working/outputs') / file_name,
    ]

    for candidate in direct_candidates:
        if candidate.exists() and candidate.is_file():
            return candidate

    # Fallback search: lightweight bounded traversal over common roots.
    for root in search_roots:
        if not root.exists():
            continue
        for candidate in sorted(root.glob(f'**/{file_name}')):
            if candidate.is_file() and '/outputs/' in candidate.as_posix():
                return candidate

    return None


resolved_files: list[Path] = []
missing_names: list[str] = []
for file_name in required_names:
    resolved = _resolve_metrics_file(file_name)
    if resolved is None:
        missing_names.append(file_name)
    else:
        resolved_files.append(resolved)

if missing_names:
    raise FileNotFoundError(
        'Metrics files were not found. Missing: ' + ', '.join(missing_names) + '; '
        'Run the metrics export cell first;'
    )

print('Resolved source files:')
for path in resolved_files:
    print(f'- {path};')

export_dir = Path('/kaggle/working/stage2_exports')
export_dir.mkdir(parents=True, exist_ok=True)
print(f'Export directory: {export_dir};')

for source_path in resolved_files:
    target_path = export_dir / source_path.name
    shutil.copy2(source_path, target_path)

    if not target_path.exists():
        raise RuntimeError(f'Copy failed for {target_path};')

    size_bytes = target_path.stat().st_size
    relative_for_kaggle = target_path.relative_to('/kaggle/working').as_posix()
    download_url = f'/files/{relative_for_kaggle}'

    print(f'Prepared: {target_path} | bytes={size_bytes};')
    display(HTML(f'<a href="{download_url}" target="_blank">Download {target_path.name}</a>'))

print('Done. If links still fail in your session, open the right Files sidebar and download from /kaggle/working/stage2_exports;')

In [ ]:
import os
import time
from datetime import datetime

import numpy as np

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

if 'detector' not in globals():
    raise RuntimeError('detector is missing - run Cell 3 first;')
if 'reid_model' not in globals():
    raise RuntimeError('reid_model is missing - run Cell 3 first;')
if 'tracker_config' not in globals():
    raise RuntimeError('tracker_config is missing - run Cell 3 first;')
if 'soccernet_loader' not in globals():
    raise RuntimeError('soccernet_loader is missing - run Cell 4 first;')
if 'iter_image_sequence_frames' not in globals():
    raise RuntimeError('iter_image_sequence_frames is missing - run Cell 4 first;')
if '_extract_crops_for_reid' not in globals():
    raise RuntimeError('_extract_crops_for_reid is missing - run Cell 6 first;')
if 'evaluation_sources' not in globals() or not evaluation_sources:
    raise RuntimeError('evaluation_sources is missing - run Cell 4 first;')

target_sequence_token = os.environ.get('STAGE2_GRID_TARGET_SEQUENCE', 'SNMOT-123').strip()
grid_frame_stride = int(os.environ.get('STAGE2_GRID_FRAME_STRIDE', str(globals().get('frame_stride', 2))))
grid_max_frames = int(os.environ.get('STAGE2_GRID_MAX_FRAMES', str(globals().get('max_frames_per_source', 0))))

if grid_frame_stride < 1:
    raise ValueError('STAGE2_GRID_FRAME_STRIDE must be >= 1;')
if grid_max_frames < 0:
    raise ValueError('STAGE2_GRID_MAX_FRAMES must be >= 0;')

print(f'Grid Search Setup: started at {datetime.now().isoformat(timespec="seconds")};')
print(f'Target sequence token: {target_sequence_token};')
print(f'Grid frame stride: {grid_frame_stride};')
print(f'Grid max frames: {grid_max_frames} (0 means no explicit cap);')


def _select_grid_source_record(token: str) -> dict:
    normalized_token = token.lower().strip()

    for source in evaluation_sources:
        source_name = str(source.get('source_name', '')).lower()
        if normalized_token and normalized_token in source_name:
            return source

    if normalized_token and {'dataset_root', 'split_directories', '_resolve_sequence_dir_from_name', '_build_source_record'} <= set(globals().keys()):
        for split_name in sorted(globals().get('split_directories', {}).keys()):
            candidate_name = f'{split_name}/{token}'
            sequence_dir = _resolve_sequence_dir_from_name(dataset_root, split_directories, candidate_name)
            if sequence_dir is None:
                continue
            source_record = _build_source_record(dataset_root, sequence_dir)
            if source_record is not None:
                return source_record

    return evaluation_sources[0]


grid_source = _select_grid_source_record(target_sequence_token)
grid_source_name = str(grid_source['source_name'])
grid_source_kind = str(grid_source['frame_source_kind'])
grid_gt_tracks_by_frame = grid_source['gt_tracks_by_frame']

print(f'Grid source selected: {grid_source_name} ({grid_source_kind});')
print(f'Ground-truth frames available: {len(grid_gt_tracks_by_frame)};')

if grid_source_kind == 'video':
    grid_frame_iterator = soccernet_loader.iter_video_frames(
        video_path=grid_source['source_path'],
        max_frames=(None if grid_max_frames == 0 else grid_max_frames),
        stride=grid_frame_stride,
    )
    estimated_total_frames = None
else:
    image_files = grid_source['image_files']
    if grid_max_frames > 0:
        max_frames_for_iter = grid_max_frames
        estimated_total_frames = min(grid_max_frames, max(0, len(image_files) // grid_frame_stride))
    else:
        max_frames_for_iter = None
        estimated_total_frames = max(0, len(image_files) // grid_frame_stride)
    grid_frame_iterator = iter_image_sequence_frames(
        image_files=image_files,
        max_frames=max_frames_for_iter,
        stride=grid_frame_stride,
    )

cached_tracking_inputs = []
cache_started_at = time.perf_counter()

iterator = grid_frame_iterator
if tqdm is not None:
    iterator = tqdm(
        grid_frame_iterator,
        total=estimated_total_frames,
        desc='Caching detections and embeddings',
        unit='frame',
    )

for step_index, (frame_index, frame) in enumerate(iterator, start=1):
    detections = detector.predict(frame)

    if detections:
        crops = _extract_crops_for_reid(frame, detections)
        embeddings = reid_model.extract(crops)

        bboxes_xyxy = np.asarray([det.bbox_xyxy for det in detections], dtype=np.float32)
        confidences = np.asarray([det.confidence for det in detections], dtype=np.float32)
        class_ids = np.asarray([det.class_id for det in detections], dtype=np.int32)
        embedding_matrix = np.asarray(embeddings, dtype=np.float32) if embeddings is not None else None
    else:
        bboxes_xyxy = np.zeros((0, 4), dtype=np.float32)
        confidences = np.zeros((0,), dtype=np.float32)
        class_ids = np.zeros((0,), dtype=np.int32)
        embedding_matrix = None

    cached_tracking_inputs.append(
        {
            'frame_index': int(frame_index),
            'bboxes_xyxy': bboxes_xyxy,
            'confidences': confidences,
            'class_ids': class_ids,
            'embeddings': embedding_matrix,
        }
    )

    if step_index % 50 == 0:
        print(
            f'Cache progress: frame_count={step_index}; '
            f'last_frame={int(frame_index)}; detections={int(confidences.shape[0])}; '
            f'timestamp={datetime.now().isoformat(timespec="seconds")};'
        )

if not cached_tracking_inputs:
    raise RuntimeError('No cached frames were created for grid search;')

cache_seconds = time.perf_counter() - cache_started_at
cache_total_detections = int(sum(np.asarray(item['confidences']).shape[0] for item in cached_tracking_inputs))
cache_avg_detections = cache_total_detections / float(len(cached_tracking_inputs))

grid_search_context = {
    'source_name': grid_source_name,
    'source_kind': grid_source_kind,
    'gt_tracks_by_frame': grid_gt_tracks_by_frame,
    'cached_tracking_inputs': cached_tracking_inputs,
    'cache_seconds': float(cache_seconds),
    'frame_count': int(len(cached_tracking_inputs)),
    'avg_detections_per_frame': float(cache_avg_detections),
}

print('Caching Summary: ready for association-only grid search;')
print(f'- Source: {grid_source_name};')
print(f'- Frames cached: {len(cached_tracking_inputs)};')
print(f'- Total detections: {cache_total_detections};')
print(f'- Average detections per frame: {cache_avg_detections:.2f};')
print(f'- Cache runtime seconds: {cache_seconds:.2f};')

In [ ]:
import gc
import itertools
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from models.trackers import ByteTrackTracker
from utils import compute_mot_id_metrics

try:
    import torch
except Exception:
    torch = None

if 'grid_search_context' not in globals():
    raise RuntimeError('grid_search_context is missing - run Cell 9 first;')
if 'tracker_config' not in globals():
    raise RuntimeError('tracker_config is missing - run Cell 3 first;')

association_alpha_grid = [0.7, 0.8, 0.9, 0.95, 0.98]
match_threshold_grid = [0.75, 0.80, 0.85]
requested_reid_dist_threshold_grid = [0.2, 0.4, 0.6]

supports_reid_dist_threshold = 'reid_dist_threshold' in tracker_config
effective_reid_dist_threshold_grid = requested_reid_dist_threshold_grid if supports_reid_dist_threshold else [None]

if supports_reid_dist_threshold:
    print('Parameter Support: reid_dist_threshold is supported by tracker config;')
else:
    print('Parameter Support: reid_dist_threshold is not supported by tracker config and will be logged as null;')

grid_inputs = grid_search_context['cached_tracking_inputs']
grid_gt_tracks_by_frame = grid_search_context['gt_tracks_by_frame']
grid_source_name = grid_search_context['source_name']

parameter_grid = list(itertools.product(association_alpha_grid, match_threshold_grid, effective_reid_dist_threshold_grid))
print(f'Grid Search Plan: source={grid_source_name}; combinations={len(parameter_grid)};')

results_rows: list[dict] = []
overall_started_at = time.perf_counter()

for run_index, (association_alpha, match_threshold, reid_dist_threshold) in enumerate(parameter_grid, start=1):
    iteration_started_at_iso = datetime.now().isoformat(timespec='seconds')
    iteration_started_at = time.perf_counter()

    print(
        f'Run Start: {run_index}/{len(parameter_grid)}; '
        f'started_at={iteration_started_at_iso}; '
        f'association_alpha={association_alpha:.2f}; '
        f'match_threshold={match_threshold:.2f}; '
        f'reid_dist_threshold={reid_dist_threshold};'
    )

    local_tracker_config = dict(tracker_config)
    local_tracker_config['association_alpha'] = float(association_alpha)
    local_tracker_config['motion_weight'] = float(association_alpha)
    local_tracker_config['embedding_weight'] = float(1.0 - association_alpha)
    local_tracker_config['match_threshold'] = float(match_threshold)

    if reid_dist_threshold is not None:
        local_tracker_config['reid_dist_threshold'] = float(reid_dist_threshold)

    tracker_local = ByteTrackTracker.from_config(local_tracker_config)
    tracker_local.initialize()

    predictions_by_frame: dict[int, list] = {}
    processed_frame_ids: list[int] = []
    frame_latencies: list[float] = []

    for frame_step, cached in enumerate(grid_inputs, start=1):
        frame_started_at = time.perf_counter()
        tracks = tracker_local.update_from_arrays(
            bboxes_xyxy=np.asarray(cached['bboxes_xyxy'], dtype=np.float32),
            confidences=np.asarray(cached['confidences'], dtype=np.float32),
            class_ids=np.asarray(cached['class_ids'], dtype=np.int32),
            embeddings=(
                None
                if cached['embeddings'] is None
                else np.asarray(cached['embeddings'], dtype=np.float32)
            ),
            frame_index=int(cached['frame_index']),
        )
        frame_latencies.append(time.perf_counter() - frame_started_at)

        frame_id = int(cached['frame_index'])
        predictions_by_frame[frame_id] = tracks
        processed_frame_ids.append(frame_id)

        if frame_step % 50 == 0 or frame_step == len(grid_inputs):
            partial_gt = {frame_id: grid_gt_tracks_by_frame.get(frame_id, []) for frame_id in processed_frame_ids}
            partial_pred = {frame_id: predictions_by_frame[frame_id] for frame_id in processed_frame_ids}
            partial_metrics = compute_mot_id_metrics(
                ground_truth_by_frame=partial_gt,
                predictions_by_frame=partial_pred,
                iou_threshold=float(globals().get('iou_threshold', 0.5)),
            )
            partial_fps = (1.0 / float(np.mean(frame_latencies))) if frame_latencies else 0.0
            print(
                f'Run Progress: {run_index}/{len(parameter_grid)}; '
                f'frames={frame_step}; '
                f'partial_id_swaps={int(partial_metrics["ID Swaps"])}; '
                f'partial_fps={partial_fps:.2f}; '
                f'timestamp={datetime.now().isoformat(timespec="seconds")};'
            )

    final_gt = {frame_id: grid_gt_tracks_by_frame.get(frame_id, []) for frame_id in processed_frame_ids}
    final_metrics = compute_mot_id_metrics(
        ground_truth_by_frame=final_gt,
        predictions_by_frame=predictions_by_frame,
        iou_threshold=float(globals().get('iou_threshold', 0.5)),
    )
    final_fps = (1.0 / float(np.mean(frame_latencies))) if frame_latencies else 0.0
    iteration_seconds = time.perf_counter() - iteration_started_at
    iteration_finished_at_iso = datetime.now().isoformat(timespec='seconds')

    results_rows.append(
        {
            'run_index': int(run_index),
            'source_name': grid_source_name,
            'association_alpha': float(association_alpha),
            'match_threshold': float(match_threshold),
            'reid_dist_threshold': (None if reid_dist_threshold is None else float(reid_dist_threshold)),
            'MOTA': float(final_metrics['MOTA']),
            'IDF1': float(final_metrics['IDF1']),
            'ID Swaps': int(final_metrics['ID Swaps']),
            'FPS': float(final_fps),
            'processed_frames': int(len(processed_frame_ids)),
            'started_at': iteration_started_at_iso,
            'finished_at': iteration_finished_at_iso,
            'duration_seconds': float(iteration_seconds),
        }
    )

    print(
        f'Run End: {run_index}/{len(parameter_grid)}; '
        f'finished_at={iteration_finished_at_iso}; '
        f'MOTA={float(final_metrics["MOTA"]):.4f}; '
        f'IDF1={float(final_metrics["IDF1"]):.4f}; '
        f'ID_Swaps={int(final_metrics["ID Swaps"])}; '
        f'FPS={final_fps:.2f}; '
        f'duration_seconds={iteration_seconds:.2f};'
    )

    tracker_local.reset()
    tracker_local.shutdown()
    del tracker_local
    del predictions_by_frame
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results_rows)
if results_df.empty:
    raise RuntimeError('Grid search produced no results;')

results_df = results_df.sort_values(
    by=['IDF1', 'MOTA', 'ID Swaps', 'FPS'],
    ascending=[False, False, True, False],
).reset_index(drop=True)

results_output_path = Path('/kaggle/working/stage2_grid_search_results.csv')
results_df.to_csv(results_output_path, index=False)

total_seconds = time.perf_counter() - overall_started_at
best_row = results_df.iloc[0].to_dict()

print('Grid Search Summary: completed;')
print(f'- Source: {grid_source_name};')
print(f'- Total runs: {len(results_df)};')
print(f'- Total runtime seconds: {total_seconds:.2f};')
print(f'- Saved results CSV: {results_output_path};')
print(
    f'- Best config: association_alpha={best_row["association_alpha"]:.2f}; '
    f'match_threshold={best_row["match_threshold"]:.2f}; '
    f'reid_dist_threshold={best_row["reid_dist_threshold"]}; '
    f'IDF1={best_row["IDF1"]:.4f}; '
    f'MOTA={best_row["MOTA"]:.4f}; '
    f'ID_Swaps={int(best_row["ID Swaps"])}; '
    f'FPS={best_row["FPS"]:.2f};'
)

display(results_df)

alpha_idf1_summary = results_df.groupby('association_alpha', as_index=False).agg({'IDF1': 'max'})
alpha_idf1_summary = alpha_idf1_summary.sort_values(by=['association_alpha'])

plt.figure(figsize=(8, 5))
plt.plot(alpha_idf1_summary['association_alpha'], alpha_idf1_summary['IDF1'], marker='o')
plt.title('Sensitivity Analysis: association_alpha vs max IDF1')
plt.xlabel('association_alpha')
plt.ylabel('max IDF1')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()